In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
import datetime

# Load the data
train_data = pd.read_csv('train.csv')
train_data.drop('w', axis=1, inplace=True)
test_data = pd.read_csv('test.csv')

In [2]:
train_data.head()

,y,square_meters,bathrooms_number,lift,rooms_number,other_features,total_floors_in_building,car_parking,availability,condominium_fees,year_of_construction,conditions,zone,floor,heating_centralized,energy_efficiency_class
0,1125000,135,2,yes,3,optic fiber | security door | balcony | full d...,7,no,available,417,NaN,excellent / refurbished,quadronno - crocetta,2,central,g
1,500000,57,1,yes,2,optic fiber | video entryphone | alarm system ...,7,1 in garage/box,available,No condominium fees,2010.0,excellent / refurbished,porta romana - medaglie d'oro,1,central,e
2,395000,92,1,yes,3,electric gate | optic fiber | video entryphone...,8,no,available,200,1960.0,excellent / refurbished,gallaratese,1,central,e
3,340000,63,1,no,2,optic fiber | video entryphone | security door...,8,no,NaN,208,NaN,excellent / refurbished,martini - insubria,mezzanine,central,f
4,199000,38,1,no,1,optic fiber | security door | internal exposur...,4,no,available,80,1930.0,excellent / refurbished,navigli - darsena,1,independent,g


In [3]:
test_data.head()

,square_meters,bathrooms_number,lift,rooms_number,other_features,total_floors_in_building,car_parking,availability,condominium_fees,year_of_construction,conditions,zone,floor,heating_centralized,energy_efficiency_class
0,112,1,yes,3,alarm system | balcony | full day concierge | ...,6,no,available,250,1965.0,good condition / liveable,certosa,2,central,g
1,180,3,yes,5,electric gate | optic fiber | video entryphone...,7,2 in garage/box,available,410,1960.0,excellent / refurbished,carrobbio,4,central,NaN
2,92,2,yes,2,electric gate | attic | optic fiber | video en...,5,no,available,500,1940.0,excellent / refurbished,brera,5,independent,g
3,61,1,yes,2,optic fiber | security door | external exposur...,5,no,NaN,100,1900.0,good condition / liveable,paolo sarpi,2,independent,d
4,130,2,yes,3,fireplace | optic fiber | alarm system | secur...,8,no,available,308,1970.0,good condition / liveable,frua,4,central,e


In [4]:
def encoder(train_data):
    # bathrooms_number
    train_data.bathrooms_number = train_data.bathrooms_number.map({'2':2, '1':1, '3':3, '3+':4,np.nan: (1+2+3+4)/4}) # (1+2+3+4)/4

    # lift from yes/no to 1/0
    train_data.lift = train_data.lift.map({'yes':1, 'no':0, np.nan: 0.5}) # 0.5

    # rooms_number
    train_data.rooms_number = train_data.rooms_number.map({'3':3, '2':2, '1':1, '5+':6, '4':4, '5':5})

    # conditions
    train_data.conditions = train_data.conditions.map({'excellent / refurbished': 1, 'good condition / liveable': 3,
           'to be refurbished': 4, 'new / under construction': 2, np.nan:(1+2+3+4)/4}) # (1+2+3+4)/4

    # floor
    train_data.floor= train_data.floor.map({'2': 2, '1': 1, 'mezzanine': 0.5, '5': 5, '6': 6, 'semi-basement': -0.5, 'ground floor': 0,
           '3': 3, '8': 8, '7': 7, '4': 4, '9': 9})

    # heating_centralized
    train_data.heating_centralized = train_data.heating_centralized.map({'central':0, 'independent':1, np.nan:0.5}) # 0.5

    # energy_efficiency_class
    train_data.energy_efficiency_class = train_data.energy_efficiency_class.map({'a':1, 'b':2, 'g':7, 'e':5, 'f':6, 'd':4, 'c':3, ',':0, np.nan:np.round((0+1+2+3+4+5+6+7)/8,1)}) # np.round((0+1+2+3+4+5+6+7)/8,1)

    # availability
    available = []
    availability = []
    dates = []

    for element in train_data.availability:
        if isinstance(element, str):
            element = element.split(' ')
            if len(element) > 1:
                dates.append(element[-1])

    # Convert date strings to datetime objects
    date_objects = []
    for date_str in dates:
        # Convert the string to a datetime object
        date_object = datetime.datetime.strptime(date_str, '%d/%m/%Y')
        date_objects.append(date_object)

    # Find the minimum date
    min_date = min(date_objects)
    min_d = min_date.strftime('%d/%m/%Y')

    for element in train_data.availability:
        if isinstance(element, str):
            if element == 'available':
                available.append(min_d)
            else:
                element = element.split(' ')
                available.append(element[-1])
        else:
            available.append(np.nan)

    timestamps = []
    date_integers = []
    for element in available:
        if isinstance(element, str):
            timestamps.append(int(datetime.datetime.strptime(element, '%d/%m/%Y').timestamp()))
            date_integers.append(int(datetime.datetime.strptime(element, '%d/%m/%Y').strftime('%Y%m%d')))
        else:
            timestamps.append(np.nan)
            date_integers.append(np.round(2956056093/146,1)) # np.round(2956056093/146,1)

    train_data['availability'] = date_integers

    # car_parking, new columns, 1 for garage/box, 1 for shared_parking
    garage_box = []
    shared_parking = []
    for element in train_data.car_parking:
        if element == np.nan:
            shared_parking.append(0.5) # 0.5
            garage_box.append(0.5) # 0.5
        if element == 'no':
            garage_box.append(0)
            shared_parking.append(0)
        element = element.split(' ')
        if 'shared' in element and 'garage/box' in element:
            shared_parking.append(element[3])
            garage_box.append(element[0])
        else:
            if 'shared' in element:
                shared_parking.append(element[0])
                garage_box.append(0)
            elif 'garage/box' in element:
                shared_parking.append(0)
                garage_box.append(element[0])

    train_data.drop('car_parking', axis=1, inplace=True)
    train_data['garage/box'] = garage_box
    train_data['shared_parking'] = shared_parking
    
    # total_floors_in_building
    t = []
    for element in train_data.total_floors_in_building:
        if element == '1 floor':
            t.append(1)
        elif element == np.nan:
            t.append(np.round(260/22,1))
        else:
            t.append(element)

    train_data['total_floors_in_building'] = t

    train_data.total_floors_in_building = train_data.total_floors_in_building.map({'7':7, '8':8, '4':4, '5':5, '6':6, '3':3, 1:1, '2':2, '23':23, '14':14, '10':10, '15':15, np.nan:np.round(260/22,1), # np.round(260/22,1)
           '13':13, '16':16, '11':11, '9':9, '12':12, '22':22, '21':21, '19':19, '27':27, '17':17, '18':18,
           '24':24})
    
    # conduminium_fees
    condominium_fees = []
    for element in train_data.condominium_fees:
        if isinstance(element, str):
            if element == 'No condominium fees':
                condominium_fees.append(0)
            else:
                condominium_fees.append(int(element))
        else:
            condominium_fees.append(np.round(743253/332,1)) # np.round(743253/332,1)

    s = sum(set(condominium_fees))
    l = len(set(condominium_fees))

    train_data['condominium_fees'] = condominium_fees
    
    # year_of_construction
    y = train_data.year_of_construction
    nan_indices = np.where(np.isnan(train_data.year_of_construction))[0]

    for idx in nan_indices:
        y[idx] = np.round(258543/134,1) # np.round(258543/134,1)


    train_data['year_of_construction'] = y
    
    # other_features
    unique_other_features = set()
    for element in train_data.other_features:
        if isinstance(element, str):
            element = element.split(' | ')
            for e in element:
                unique_other_features.add(e)

    # add new column for every other feature
    balconies = []
    alarm_system = []
    attic = []
    cellar = []
    centralized_tv_system = []
    closet = []
    disabled_access = []
    electric_gate = []
    exposure = []
    glass = []
    double_glass = []
    triple_glass = []
    fireplace = []
    concierge = []
    furnished = []
    hydromassage = []
    kitchen = []
    optic_fiber = []
    pool = []
    garden = []
    reception = []
    security_door = []
    tv_system = []
    tavern = []
    tennis_court = []
    terrace = []
    video_entryphone = []
    for e in train_data.other_features:
        if isinstance(e, str):
            # balconies
            if 'balconies' in e or 'balcony' in e:
                balconies.append(1)
            else:
                balconies.append(0)

            # alarm system
            if 'alarm system' in e:
                alarm_system.append(1)
            else:
                alarm_system.append(0)

            # attic
            if 'attic' in e:
                attic.append(1)
            else:
                attic.append(0)

            # cellar
            if 'cellar' in e:
                cellar.append(1)
            else:
                cellar.append(0)

            # centralized tv system
            if 'centralized tv system' in e:
                centralized_tv_system.append(1)
            else:
                centralized_tv_system.append(0)

            # closet
            if 'closet' in e:
                closet.append(1)
            else:
                closet.append(0)

            # disabled access
            if 'disabled access' in e:
                disabled_access.append(1)
            else:
                disabled_access.append(0)

            # electric gate
            if 'electric gate' in e:
                electric_gate.append(1)
            else:
                electric_gate.append(0)

            # exposure, count the number of exposures
            if 'exposure' in e:
                exposure.append(1)
            else:
                exposure.append(0)

            # fireplace
            if 'fireplace' in e:
                fireplace.append(1)
            else:
                fireplace.append(0)

            # concierge, full 1 or half 0.5
            if 'full day concierge' in e:
                concierge.append(1)
            elif 'half-day concierge' in e:
                concierge.append(0.5)
            else:
                concierge.append(0)

            # furnished 1, if partially 0.5
            if 'furnished' in e:
                if 'partially' in e:
                    furnished.append(0.5)
                else:
                    furnished.append(1)
            else:
                furnished.append(0)

            # hydromassage
            if 'hydromassage' in e:
                hydromassage.append(1)
            else:
                hydromassage.append(0)

            # kitchen
            if 'kitchen' in e:
                kitchen.append(1)
            else:
                kitchen.append(0)

            # optic fiber
            if 'optic fiber' in e:
                optic_fiber.append(1)
            else:
                optic_fiber.append(0)

            # pool
            if 'pool' in e:
                pool.append(1)
            else:
                pool.append(0)

            # private 1 and shared 0.5 garden
            if 'garden' in e:
                if 'private' in e and 'shared' in e:
                    garden.append(1.5)
                elif 'private' in e:
                    garden.append(1)
                elif 'shared' in e:
                    garden.append(0.5)
            else:
                garden.append(0)

            # reception
            if 'reception' in e:
                reception.append(1)
            else:
                reception.append(0)

            # security door
            if 'security door' in e:
                security_door.append(1)
            else:
                security_door.append(0)

            # tv system
            if 'tv system' in e:
                tv_system.append(1)
            else:
                tv_system.append(0)

            # tavern
            if 'tavern' in e:
                tavern.append(1)
            else:
                tavern.append(0)

            # tennis court   
            if 'tennis court' in e:
                tennis_court.append(1)
            else:
                tennis_court.append(0)

            # terrace
            if 'terrace' in e:
                terrace.append(1)
            else:
                terrace.append(0)

            # video entryphone
            if 'video entryphone' in e:
                video_entryphone.append(1)
            else:
                video_entryphone.append(0)

            # glass, double_glass, triple_glass
            if 'window frames in glass' in e:
                glass.append(1)
            else: 
                glass.append(0)
            if 'window frames in double glass' in e:
                double_glass.append(1)
            else:
                double_glass.append(0)
            if 'window frames in triple glass' in e:
                triple_glass.append(1)
            else:
                triple_glass.append(0)



        else:
            balconies.append(0.5)
            alarm_system.append(0.5)
            attic.append(0.5)
            cellar.append(0.5)
            centralized_tv_system.append(0.5)
            closet.append(0.5)
            disabled_access.append(0.5)
            electric_gate.append(0.5)
            exposure.append(0.5)
            glass.append(0.5)
            double_glass.append(0.5)
            triple_glass.append(0.5)
            fireplace.append(0.5)
            concierge.append(0.5)
            furnished.append(0.5)
            hydromassage.append(0.5)
            kitchen.append(0.5)
            optic_fiber.append(0.5)
            pool.append(0.5)
            garden.append(0.5)
            reception.append(0.5)
            security_door.append(0.5)
            tv_system.append(0.5)
            tavern.append(0.5)
            tennis_court.append(0.5)
            terrace.append(0.5)
            video_entryphone.append(0.5)

    train_data['balconies'] = balconies
    train_data['alarm_system'] = alarm_system
    train_data['attic'] = attic
    train_data['cellar'] = cellar
    train_data['centralized_tv_system'] = centralized_tv_system
    train_data['closet'] = closet
    train_data['disabled_access'] = disabled_access
    train_data['electric_gate'] = electric_gate
    train_data['exposure'] = exposure
    train_data['glass'] = glass
    train_data['double_glass'] = double_glass
    train_data['triple_glass'] = triple_glass
    train_data['glass'] = glass
    train_data['fireplace'] = fireplace
    train_data['concierge'] = concierge
    train_data['furnished'] = furnished
    train_data['hydromassage'] = hydromassage
    train_data['kitchen'] = kitchen
    train_data['optic_fiber'] = optic_fiber
    train_data['pool'] = pool
    train_data['garden'] = garden
    train_data['reception'] = reception
    train_data['security_door'] = security_door
    train_data['tv_system'] = tv_system
    train_data['tavern'] = tavern
    train_data['tennis_court'] = tennis_court
    train_data['terrace'] = terrace
    train_data['video_entryphone'] = video_entryphone

    train_data.drop('other_features', axis=1, inplace=True)
    
    # One-hot encode the 'Zone' column and convert to integer type (1 or 0)
    zone_dummies = pd.get_dummies(train_data['zone'], prefix='zone', dtype=int)

    # Concatenate the original DataFrame with the new dummy variables
    train_data = pd.concat([train_data, zone_dummies], axis=1)

    # Optionally, drop the original 'Zone' column if it's no longer needed
    train_data.drop('zone', axis=1, inplace=True)

    # Display the first few rows of the updated DataFrame
    train_data.head()
    
    if 'y' in train_data.columns:
        # only for train
        missing_features_train = ['zone_corso magenta', 'zone_largo caioroli 2', 'zone_via marignano, 3']

        for element in missing_features_train:
            if element not in train_data.columns:
                train_data[element] = 0

        # only for train
        # Columns you want to move to the end
        columns_to_move = ['zone_cascina gobba', 'zone_scala - manzoni', "zone_via fra' cristoforo"]

        # Create a new list of columns with the selected columns at the end
        new_order = [col for col in train_data.columns if col not in columns_to_move] + columns_to_move

        # Reorder the columns in the DataFrame
        train_data = train_data[new_order]
    else:
        # only for test
        # Columns you want to move to the end
        columns_to_move = ['zone_corso magenta', 'zone_largo caioroli 2', 'zone_via marignano, 3']

        # Create a new list of columns with the selected columns at the end
        new_order = [col for col in train_data.columns if col not in columns_to_move] + columns_to_move

        # Reorder the columns in the DataFrame
        train_data = train_data[new_order]

        # only for test
        missing_features_test = ['zone_cascina gobba', 'zone_scala - manzoni', "zone_via fra' cristoforo"]

        for element in missing_features_test:
            if element not in train_data.columns:
                train_data[element] = 0
    
    # Check if there are any NaN values in the entire DataFrame
    has_nan = train_data.isna().sum() 
    print("Are there any NaN values in the DataFrame?", has_nan)
    
    frequencies = {col: train_data[col].value_counts() for col in train_data.columns}
    print('Frequencies : ', frequencies)
    
    
    return train_data

In [ ]:
# both will give an error but it is only a warning
train_data = encoder(train_data)
train_data.to_csv('Encoded_Train.csv', index=False)

In [ ]:
test_data = encoder(test_data)
test_data.to_csv('Encoded_Test.csv', index=False)